# Week 2: Similarity, Clustering, and Stable Patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/02/Week_02_Similarity_Clustering_Stable_Patterns.ipynb)

**Advanced Data Mining · Master level · 180-minute core session**

Last week, cleaning choices changed our descriptive findings. Today, representation choices change who appears similar to whom. **Does a convincing cluster plot survive a reasonable change of assumptions?**

## Learning goals

You will construct customer-level features, explain how units and scaling affect distance, compare k-means with a density-based method on a controlled example, inspect real customer groups, and distinguish internal separation from stability and usefulness.

**Recap:** retain raw data; name the analysis unit; make policies explicit; report sensitivity. From the previous course, remember that a representation preserves some differences and suppresses others.

Core route: distance intuition → customer features → coding block 1 (40 min) → clustering and projection → coding block 2 (40 min) → conclusions. This notebook runs independently of Week 1; it rebuilds the same positive-sales policy.

## Environment and getting started

**Colab:** click **Open in Colab** above, connect to a CPU runtime, and choose **Runtime → Run all**. The setup cell installs the required packages; the data cell downloads and verifies the small course dataset automatically. No upload, ZIP extraction, account token, or manual data preparation is required. If Colab asks for a runtime restart after installation, restart and choose Run all again.

**Local:** from the course root, run `uv sync --locked`, then `uv run jupyter lab`. Open this notebook with the environment's Python kernel and run cells in order. The supplied `data/` file is used locally; otherwise the same download runs automatically. First-time installation/download needs internet. Afterward, local cached data is reused.

**Prerequisites:** Python variables, pandas tables, basic plots, and the previous course's idea that a representation determines what information is available. CPU is sufficient; no neural-network training is required today.

Run the baseline first. During the investigations, change one setting and rerun its relevant cells. Restart and run all if changes leave you unsure which variables are current.

In [ ]:
# Install automatically in Colab; local environments are managed with uv.
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'numpy>=2,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8'
    ])
    print('Colab dependencies ready. Continue with the next cell.')
else:
    print('Local runtime: using the course environment (uv sync --locked).')

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
pd.set_option('display.max_columns', 12)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
print('NumPy:', np.__version__, '| pandas:', pd.__version__)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from sklearn.datasets import make_moons
import sklearn
print('scikit-learn:', sklearn.__version__)

## 1. Similarity is a choice

For vectors $x,y$, Euclidean distance is $d(x,y)=\sqrt{\sum_j(x_j-y_j)^2}$. A large numerical difference contributes more, regardless of whether it is measured in days or pounds. Converting pounds to pence can change nearest neighbors even though customer behavior did not change.

Standardization uses $z_j=(x_j-\mu_j)/\sigma_j$, with mean and standard deviation estimated from the reference data. **ELI5:** we first put each ruler into comparable units. This does not prove the features deserve equal importance, and outliers can still affect the ruler.

**Pause and predict:** for a customer with 5 orders and £500 spend, is a customer with 6 orders and £100 spend nearer than one with 10 orders and £480 spend? Does that answer depend on the ruler?

In [ ]:
toy_customers = pd.DataFrame({'orders': [5, 6, 10, 2],
                              'spend_GBP': [500, 100, 480, 200]},
                             index=['query', 'A', 'B', 'C'])
toy_scaled = StandardScaler().fit_transform(toy_customers)
distances = pd.DataFrame({
    'raw_distance': euclidean_distances(toy_customers.iloc[[0]], toy_customers)[0],
    'standardized_distance': euclidean_distances(toy_scaled[[0]], toy_scaled)[0]}, index=toy_customers.index)
display(toy_customers); display(distances.drop('query').round(3))
# TODO 1: Multiply spend_GBP by 100 before calculating both distances.
# Predict which ranking should be invariant, then test your prediction.

### Direction versus magnitude

Cosine similarity is $\frac{x^\top y}{\|x\|\|y\|}$; cosine distance is one minus this value. For nonzero purchase-category vectors, it compares the pattern of spending while ignoring a positive common scale factor. It can therefore consider a small and a large customer identical in direction.

This is useful when category preference is the question, but loses total volume. Applying cosine distance to arbitrarily centered features can change its meaning. Today we compare it on a simple nonnegative example; customer clustering later uses Euclidean distance on transformed features.

In [ ]:
profiles = pd.DataFrame([[1, 2, 0], [10, 20, 0], [0, 2, 1]],
                        index=['small', 'same_mix_large', 'different_mix'],
                        columns=['books', 'food', 'tools'])
display(pd.DataFrame({'euclidean_from_small': euclidean_distances(profiles.iloc[[0]], profiles)[0],
                      'cosine_from_small': cosine_distances(profiles.iloc[[0]], profiles)[0]},
                     index=profiles.index).round(3))
# TODO 2: Change the large customer's category mix. Explain what each distance notices.

### Our real dataset: a reproducible retail extract

We use a supplied extract of [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), a UK online retailer's historical invoice lines. A row is a **product line**, not a customer and not necessarily an entire purchase. Prices are in pounds sterling; invoice IDs beginning with `C` mark cancellations.

The extract preserves every source row for 400 sampled customer IDs, plus rows from ten sampled anonymous invoices per calendar month. It retains source defects and cancellations. Known and anonymous records have different sampling mechanisms: their relative proportions cannot estimate missing-ID prevalence in the original population. This is a historical case study, not a claim about present-day shoppers. December 2011 is incomplete.

**Attribution:** Chen, D. (2015), *Online Retail*, UCI, [DOI: 10.24432/C5BW33](https://doi.org/10.24432/C5BW33), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course adaptation is customer/invoice selection plus CSV conversion; see `data/README.md` and `retail_manifest.json` for provenance. No synthetic defects were added to the real extract.

In [ ]:
# Prefer the bundled/local cache; otherwise fetch the public course dataset.
import urllib.request
import urllib.error

filename = 'online_retail_teaching.csv.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/online_retail_teaching.csv.gz'
expected_sha256 = '8503a734cd203ed63df3292c2a12d0b803206dc18ed67a7dc83664fd307d96d8'
candidates = [Path.cwd() / filename]
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(folder / 'data' / filename)
data_path = next((p for p in candidates if p.is_file()), None)

if data_path is None:
    data_path = Path.cwd() / 'data' / filename
    print('Downloading the course dataset (approximately 610 KB)...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            payload = response.read()
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(
            'Could not download the course dataset. Check your internet connection and retry. '
            'If the URL is unavailable, ask the instructor to check the public course release. '
            f'URL: {DATA_URL}'
        ) from error
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError('Downloaded data does not match this notebook version. Reopen the current course notebook.')
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(payload)
else:
    print('Using the cached course dataset.')

if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Cached dataset differs from this notebook version. Use a fresh Colab runtime or restore the matching course data locally.')
raw = pd.read_csv(data_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string',
                                  'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
assert len(raw) > 0 and raw.InvoiceDate.notna().all()
print(f'{len(raw):,} rows; {raw.CustomerID.nunique()} observed customer IDs')
display(raw.head(4))

## 2. A customer representation: recency, frequency, monetary value

We now change the observation unit from invoice lines to customers. **Recency** is days since the last positive purchase; **frequency** counts distinct positive invoices; **monetary value** sums positive recorded line values. These describe observed behavior within the dataset window, not lifetime value or profitability.

We exclude anonymous rows because they cannot be assigned to customers. We drop exact repeats as an explicit policy, exclude cancellations and nonpositive quantity/price, and fix the observation endpoint at **10 December 2011**, the day after the source window ends. Using today's date would make this historical exercise drift over time.

Frequency counts invoices, not product lines. Returns are not subtracted from monetary value under this policy. A high-spend customer may also have substantial returns; that is an extension, not a hidden interpretation of our feature.

In [ ]:
sales = raw.drop_duplicates().copy()
sales = sales[~sales.InvoiceNo.str.upper().str.startswith('C', na=False)]
sales = sales[(sales.Quantity > 0) & (sales.UnitPrice > 0) & sales.CustomerID.notna()].copy()
sales['LineValue'] = sales.Quantity * sales.UnitPrice
snapshot = pd.Timestamp('2011-12-10')
rfm = sales.groupby('CustomerID').agg(
    last_purchase=('InvoiceDate', 'max'), frequency=('InvoiceNo', 'nunique'),
    monetary=('LineValue', 'sum'))
rfm['recency'] = (snapshot - rfm.last_purchase).dt.days
rfm = rfm[['recency', 'frequency', 'monetary']].sort_index()
assert (rfm >= 0).all().all() and rfm.index.is_unique
display(rfm.describe().round(2))
print('Customers with positive identified sales:', len(rfm))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, column in zip(axes, rfm.columns):
    ax.hist(rfm[column], bins=25, color='#277da8')
    ax.set(xlabel=column, ylabel='Customer count')
plt.tight_layout(); plt.show()
# TODO 3: Inspect the top five monetary values. Describe the long tail without deleting it.
display(rfm.nlargest(5, 'monetary'))

## 3. Coding block 1 — change the representation (40 minutes)

We compare raw features, standardized features, and `log1p` followed by standardization. The log compresses large differences: moving from £10 to £100 becomes more comparable to moving from £100 to £1,000. That is a change in what we consider important, not a neutral cleanup.

For a customer nearest the per-feature medians in standardized space, inspect its five nearest neighbors under each representation. We choose this query deterministically so everyone compares the same case. Customer IDs are retained only as labels and never included as numeric features.

**Pause and predict:** which feature will dominate raw distances? Can standardized and log-standardized distances disagree even though both remove unit differences?

In [ ]:
X_raw = rfm.to_numpy(dtype=float)
X_standard = StandardScaler().fit_transform(X_raw)
X_log = StandardScaler().fit_transform(np.log1p(X_raw))
spaces = {'raw': X_raw, 'standardized': X_standard, 'log_standardized': X_log}
median_position = np.median(X_standard, axis=0)
query_position = np.argmin(np.linalg.norm(X_standard - median_position, axis=1))
query_id = rfm.index[query_position]
print('Query ID:', query_id)
display(rfm.loc[[query_id]])

neighbor_sets = {}
for name, X in spaces.items():
    distance = np.linalg.norm(X - X[query_position], axis=1)
    distance[query_position] = np.inf
    positions = np.argsort(distance)[:5]
    neighbor_sets[name] = set(rfm.index[positions])
    print(name)
    display(rfm.iloc[positions].assign(distance=distance[positions]).round(2))

In [ ]:
# TODO 4: Choose another query by changing query_position in the cell above.
# TODO 5: Give monetary value double weight in standardized log space below.
weights = np.array([1., 1., 2.])
X_weighted = X_log * weights
d = np.linalg.norm(X_weighted - X_weighted[query_position], axis=1)
d[query_position] = np.inf
weighted_neighbors = set(rfm.index[np.argsort(d)[:5]])
overlap = len(weighted_neighbors & neighbor_sets['log_standardized']) / 5
print('Fraction of previous five neighbors retained:', overlap)
display(rfm.loc[sorted(weighted_neighbors)])

### Representation report

Write 100–150 words: name your query; describe one changed neighbor in original units; explain the transformation responsible; and state which business question makes that transformation reasonable. If neighbors do not change, report that honestly and try a second query.

**Common confusion:** multiplying a feature by two makes its contribution to **squared** Euclidean distance four times as large. “Double importance” needs a precise definition.

Scaling statistics were fitted to this descriptive cohort. For future-customer evaluation, fit transformations on the reference/training cohort and reuse them unchanged on held-out customers.

## 4. Algorithms encode assumptions

K-means partitions all observations into a chosen $k$ groups by minimizing within-cluster squared Euclidean distances: $\sum_i\|x_i-\mu_{c_i}\|^2$. It tends to work well for compact groups in the chosen geometry. It always assigns a group, even when the data contains no useful segmentation.

DBSCAN instead joins dense neighborhoods, controlled by a radius `eps` and a minimum neighborhood count `min_samples`. It can identify curved groups and label sparse observations as noise (`-1`). A noise label is not evidence of error or fraud. Both methods depend on representation and parameters.

The synthetic moons below isolate shape from business interpretation. They are not a model of retail customers. Predict which algorithm will follow the curves before running it.

In [ ]:
moons, true_moon = make_moons(n_samples=300, noise=.06, random_state=SEED)
moon_kmeans = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(moons)
moon_dbscan = DBSCAN(eps=.2, min_samples=5).fit_predict(moons)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, labels, title in zip(axes, [true_moon, moon_kmeans, moon_dbscan],
                           ['Known synthetic groups', 'K-means: compact groups', 'DBSCAN: density connectivity']):
    ax.scatter(moons[:, 0], moons[:, 1], c=labels, cmap='tab10', s=18)
    ax.set_title(title); ax.set_xlabel('x1'); ax.set_ylabel('x2')
plt.tight_layout(); plt.show()
# TODO 6: Change eps to 0.1 and 0.4. Count clusters and noise after each run.
print('DBSCAN clusters:', len(set(moon_dbscan) - {-1}), 'noise points:', (moon_dbscan == -1).sum())

## 5. Real customer groups and a projection

For retail we start with three k-means groups in log-standardized RFM space. Three is an experimental baseline, not a discovered number of customer types. Inspect group sizes and feature medians in original units before inventing names.

PCA projects onto directions of greatest variance. A two-dimensional picture may hide distinctions present in the third dimension. We cluster in the full three-dimensional representation and only then project for display; the plot does not validate the clusters.

**ELI5:** a shadow helps you see an object, but two different objects can cast similar shadows. Report the explained-variance fraction alongside the picture.

In [ ]:
baseline = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X_log)
labels = baseline.labels_
profile = rfm.assign(cluster=labels).groupby('cluster').median()
profile['customers'] = pd.Series(labels).value_counts().sort_index()
display(profile.round(2))
pca = PCA(n_components=2)
projection = pca.fit_transform(X_log)
fig, ax = plt.subplots()
for cluster in sorted(set(labels)):
    mask = labels == cluster
    ax.scatter(projection[mask, 0], projection[mask, 1], s=22, alpha=.65, label=f'Group {cluster}')
ax.set(xlabel='Principal component 1', ylabel='Principal component 2',
       title=f'Projection retains {pca.explained_variance_ratio_.sum():.1%} of transformed variance')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Coding block 2 — can we trust the groups? (40 minutes)

We separate three questions. **Separation:** are groups compact and apart under this distance? **Stability:** do reasonable reruns or data changes preserve the partition? **Usefulness:** does the grouping support a worthwhile decision? A high score for one does not establish the others.

The silhouette for an observation is $s=(b-a)/\max(a,b)$, where $a$ is its mean within-group distance and $b$ is the smallest mean distance to another group. Higher values suggest separation under this metric. Compare values cautiously across different feature spaces because the ruler itself changes.

Adjusted Rand index (ARI) compares two partitions of the **same observations**, allowing arbitrary group numbers: 1 is identical grouping, values near 0 indicate chance-level agreement, and negative values are possible. We do not compare numeric cluster IDs directly. There are no ground-truth customer segments here.

In [ ]:
# TODO 7: Inspect k=2..6. Choose two candidates using scores AND original-unit profiles.
scores = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_log)
    scores.append({'k': k, 'silhouette': silhouette_score(X_log, model.labels_),
                   'smallest_group': np.bincount(model.labels_).min()})
display(pd.DataFrame(scores).round(3))
candidate_k = 4
candidate = KMeans(n_clusters=candidate_k, n_init=10, random_state=SEED).fit(X_log)
display(rfm.assign(cluster=candidate.labels_).groupby('cluster').agg(['median', 'size']).round(2))

In [ ]:
# Representation sensitivity: same customers, same k, different geometry.
comparisons = []
for name, X in spaces.items():
    other = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(X)
    comparisons.append({'space': name, 'ARI_vs_log_baseline': adjusted_rand_score(labels, other)})
display(pd.DataFrame(comparisons).round(3))

# Initialization sensitivity alone: fixed representation and dataset.
seed_agreement = []
for seed in range(5):
    other = KMeans(n_clusters=3, n_init=10, random_state=seed).fit_predict(X_log)
    seed_agreement.append(adjusted_rand_score(labels, other))
print('ARI across seeds:', np.round(seed_agreement, 3))

In [ ]:
# Sampling sensitivity: refit BOTH preprocessing and clustering on 80% of customers.
# Predict on the same full cohort to make each ARI comparison well-defined.
rng = np.random.default_rng(SEED)
resample_agreement = []
for repeat in range(10):
    subset = rng.choice(len(rfm), size=int(.8 * len(rfm)), replace=False)
    scaler = StandardScaler().fit(np.log1p(X_raw[subset]))
    model = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(
        scaler.transform(np.log1p(X_raw[subset])))
    predicted = model.predict(scaler.transform(np.log1p(X_raw)))
    resample_agreement.append(adjusted_rand_score(labels, predicted))
print('Resampling ARI min / median / max:', np.round([
    min(resample_agreement), np.median(resample_agreement), max(resample_agreement)], 3))
plt.plot(range(1, 11), resample_agreement, 'o-', color='#277da8')
plt.ylim(-.05, 1.05); plt.xlabel('80% subsample'); plt.ylabel('ARI against full-cohort partition')
plt.title('Descriptive sensitivity, not out-of-sample validation')
plt.tight_layout(); plt.show()
# TODO 8: Change 80% to 60% and rerun. Keep k and seed fixed to isolate sampling.

### Decision memo and wrap-up

Write 150–200 words: give your representation and proposed number of groups; characterize two groups with original-unit medians and sizes; report separation and stability evidence; and state one decision the evidence does **not** justify. “No actionable segmentation yet” is acceptable when supported.

The resampling exercise measures sensitivity relative to a full-cohort fit. Its evaluation includes fitted observations, so it is not a future-customer generalization test or a confidence interval. Strong stability also does not prove real natural categories: an algorithm can consistently partition a continuous cloud.

Discuss: could a different distance be more appropriate? Does the projection conceal overlap? Which business evidence would you collect before acting on a segment? Next week we switch from geometric similarity to items that occur together in transactions.

## Reading path

- **Start here · 20–25 min:** [Introduction to Statistical Learning with Python](https://www.statlearning.com/), Chapter 12, introductions to PCA and k-means clustering. Focus on the geometric explanations. Question: why do the units of a feature matter?
- **Go deeper · 20 min:** [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html), k-means, DBSCAN, and silhouette sections. Question: which shapes or density differences would break each method's assumptions?
- **Visual extension · 15–20 min:** [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/), examples on cluster size, distance, and parameter changes. We used PCA today; this prepares you to question nonlinear projection plots later.
- **Try it · 15 min:** repeat our moons experiment at two noise levels, recording DBSCAN's noise count. Explain which change is in the data and which is in the model.

## Homework / Post-class Extensions — optional

1. Compare k-means with DBSCAN on retail features. Report noise separately; calculate silhouette only when at least two non-noise groups and enough observations remain.
2. Inspect PCA loadings. Which transformed RFM features contribute most to each axis? Do not interpret the sign of a PCA axis as an absolute meaning.
3. Add a return-related feature using the retained raw records. Explain how it changes the observation policy and nearest neighbors.
4. Try a user–genre profile from [MovieLens Small](https://grouplens.org/datasets/movielens/). Compare cosine and Euclidean similarity. Download from the provider and follow its terms; this alternative is not bundled or required.